# Forest Cover Type — Multiclass Classification + Ensembling

Real UCI/Kaggle dataset (see `data/forest_cover/SOURCE.md`): 581,012 rows,
54 cartographic features, 7 forest cover-type classes. This notebook
demonstrates:

- Reading a bundled `.zip` (decompressed at runtime, not committed as a
  raw CSV) into Polars.
- Full EDA on a representative sample (the full `analyze()` call includes
  several O(n^2)-ish steps — PCA, VIF, causal discovery — that aren't
  meant to run on 581k rows every time; we profile a 5,000-row stratified
  sample and say so explicitly).
- A **stratified training sample** (100,000 rows) for tractable notebook
  runtime — the full bundled dataset remains available for a longer run.
- Multiclass classification with `random_forest_classifier` vs. a
  `voting_classifier` ensemble (Random Forest + Extra Trees +
  HistGradientBoosting).

Polars + numpy only — no pandas.

In [1]:
import gzip
import zipfile

import numpy as np
import polars as pl

from skyulf import EDAAnalyzer, EDAVisualizer, SkyulfPipeline

QUANTITATIVE = [
    "Elevation",
    "Aspect",
    "Slope",
    "Horizontal_Distance_To_Hydrology",
    "Vertical_Distance_To_Hydrology",
    "Horizontal_Distance_To_Roadways",
    "Hillshade_9am",
    "Hillshade_Noon",
    "Hillshade_3pm",
    "Horizontal_Distance_To_Fire_Points",
]
WILDERNESS = [f"Wilderness_Area{i}" for i in range(1, 5)]
SOIL = [f"Soil_Type{i}" for i in range(1, 41)]
COLUMNS = QUANTITATIVE + WILDERNESS + SOIL + ["Cover_Type"]

with zipfile.ZipFile("data/forest_cover/covtype.zip") as z:
    raw = gzip.decompress(z.read("covtype.data.gz"))

full = pl.read_csv(raw, has_header=False, new_columns=COLUMNS)
print(full.shape)
full.head(3)

(581012, 55)


Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area1,Wilderness_Area2,Wilderness_Area3,Wilderness_Area4,Soil_Type1,Soil_Type2,Soil_Type3,Soil_Type4,Soil_Type5,Soil_Type6,Soil_Type7,Soil_Type8,Soil_Type9,Soil_Type10,Soil_Type11,Soil_Type12,Soil_Type13,Soil_Type14,Soil_Type15,Soil_Type16,Soil_Type17,Soil_Type18,Soil_Type19,Soil_Type20,Soil_Type21,Soil_Type22,Soil_Type23,Soil_Type24,Soil_Type25,Soil_Type26,Soil_Type27,Soil_Type28,Soil_Type29,Soil_Type30,Soil_Type31,Soil_Type32,Soil_Type33,Soil_Type34,Soil_Type35,Soil_Type36,Soil_Type37,Soil_Type38,Soil_Type39,Soil_Type40,Cover_Type
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
2596,51,3,258,0,510,221,232,148,6279,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5
2590,56,2,212,-6,390,220,235,151,6225,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,5
2804,139,9,268,65,3180,234,238,135,6121,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2


## 1. Full EDA on a representative sample

We profile a 5,000-row stratified-by-class sample rather than all 581k
rows — `EDAAnalyzer.analyze()`'s PCA/VIF/causal-discovery steps are not
designed to run on hundreds of thousands of rows every call, and a 5k
stratified sample gives the same qualitative picture.

In [2]:
eda_sample = (
    full.with_columns(pl.int_range(pl.len()).over("Cover_Type").alias("_rn"))
    .filter(pl.col("_rn") < 5000 // 7)
    .drop("_rn")
)
print(eda_sample.shape)
print(eda_sample["Cover_Type"].value_counts().sort("Cover_Type"))

profile = EDAAnalyzer(eda_sample.select(QUANTITATIVE + ["Cover_Type"])).analyze(
    target_col="Cover_Type"
)
print(f"Alerts: {[a.message for a in profile.alerts][:5]}")

high_vif = {c: v for c, v in profile.vif.items() if v and v > 5}
print(f"High-VIF (>5) quantitative features: {list(high_vif.keys())}")

(4998, 55)
shape: (7, 2)
┌────────────┬───────┐
│ Cover_Type ┆ count │
│ ---        ┆ ---   │
│ i64        ┆ u32   │
╞════════════╪═══════╡
│ 1          ┆ 714   │
│ 2          ┆ 714   │
│ 3          ┆ 714   │
│ 4          ┆ 714   │
│ 5          ┆ 714   │
│ 6          ┆ 714   │
│ 7          ┆ 714   │
└────────────┴───────┘
Alerts: ["Column 'Slope' contains significant outliers.", "Column 'Horizontal_Distance_To_Hydrology' contains significant outliers.", "Column 'Vertical_Distance_To_Hydrology' contains significant outliers.", "Column 'Horizontal_Distance_To_Roadways' contains significant outliers.", "Column 'Hillshade_9am' contains significant outliers."]
High-VIF (>5) quantitative features: ['Slope', 'Hillshade_9am', 'Hillshade_Noon', 'Hillshade_3pm']


In [3]:
EDAVisualizer(profile, eda_sample.select(QUANTITATIVE + ["Cover_Type"])).summary()

╭────────────────────╮
│ Skyulf EDA Summary │
╰────────────────────╯

1. Data Quality

┏━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┓
┃ Metric         ┃ Value          ┃
┡━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━┩
│ Rows           │ 4998           │
│ Columns        │ 11             │
│ Missing Cells  │ 0.0%           │
│ Duplicate Rows │ 0              │
│ Target Column  │ Cover_Type     │
│ Task Type      │ Classification │
└────────────────┴────────────────┘

2. Numeric Statistics

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┳━━━━━━━┳━━━━━━━━━━━┓
┃ Column                             ┃    Mean ┃     Std ┃     Min ┃     Max ┃  Skew ┃  Kurt ┃ Normality ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━╇━━━━━━━╇━━━━━━━━━━━┩
│ Elevation                          │ 2682.59 │  441.11 │ 1863.00 │ 3542.00 │  0.06 │ -1.30 │    No     │
│ Aspect                             │  158.88 │  114.48 │    0.00 │  360.00 │  0.42 │ -1.32 │    No     │
│ Slope                              │   17.54 │    9.41 │    0.00 │   52.00 │  0.48 │ -0.50 │    No     │
│ Horizontal_Distance_To_Hydrology   │  214.33 │  190.87 │    0.00 │ 1159.00 │  1.49 │  3.06 │    No     │
│ Vertical_Distance_To_Hydrology     │   50.40 │   56.79 │ -134.00 │  271.00 │  1.15 │  1.11 │    No     │
│ Horizontal_Distance_To_Roadways    │ 1878.77 │ 1586.28 │   30.00 │ 6890.00 │  1.03 │ -0.02 │    No     │
│ Hillshade_9am                      │  208.88 │   34.30 │    0.00 │  254.00 │ -1.08 │  0.90 │    No     │
│ Hillshade_Noon                     │  214.58 │   25.64 │   99.00 │  254.00 │ -0.94 │  0.81 │    No     │
│ Hillshade_3pm                      │  134.05 │   51.02 │    0.00 │  248.00 │ -0.38 │ -0.25 │    No     │
│ Horizontal_Distance_To_Fire_Points │ 1584.12 │ 1350.75 │   30.00 │ 6853.00 │  1.58 │  2.38 │    No     │
└────────────────────────────────────┴─────────┴─────────┴─────────┴─────────┴───────┴───────┴───────────┘

...

• Column 'Horizontal_Distance_To_Fire_Points' contains significant outliers.

• Column 'Slope' has high VIF (7.6).

• Column 'Hillshade_9am' has very high VIF (81.6). Consider removing it.

• Column 'Hillshade_Noon' has very high VIF (31.7). Consider removing it.

• Column 'Hillshade_3pm' has very high VIF (118.8). Consider removing it.

## 2. Class balance across the full dataset

The full 581,012-row dataset is itself imbalanced across the 7 cover
types (Lodgepole/Spruce-Fir dominate); worth checking before sampling for
training.

In [4]:
print(full["Cover_Type"].value_counts().sort("Cover_Type"))

shape: (7, 2)
┌────────────┬────────┐
│ Cover_Type ┆ count  │
│ ---        ┆ ---    │
│ i64        ┆ u32    │
╞════════════╪════════╡
│ 1          ┆ 211840 │
│ 2          ┆ 283301 │
│ 3          ┆ 35754  │
│ 4          ┆ 2747   │
│ 5          ┆ 9493   │
│ 6          ┆ 17367  │
│ 7          ┆ 20510  │
└────────────┴────────┘


## 3. A stratified training sample (100,000 rows)

Kept proportional to the real class distribution above, for tractable
notebook runtime.

In [5]:
TARGET = "Cover_Type"
TRAIN_SAMPLE_SIZE = 100_000

class_counts = full[TARGET].value_counts().sort(TARGET)
total = full.height
sample_frames = []
for row in class_counts.iter_rows(named=True):
    cls, n = row[TARGET], row["count"]
    take = max(1, round(TRAIN_SAMPLE_SIZE * n / total))
    sample_frames.append(full.filter(pl.col(TARGET) == cls).sample(n=min(take, n), seed=42))

model_df = pl.concat(sample_frames).sample(fraction=1.0, shuffle=True, seed=42)
print(model_df.shape)
print(model_df[TARGET].value_counts().sort(TARGET))

(100001, 55)
shape: (7, 2)
┌────────────┬───────┐
│ Cover_Type ┆ count │
│ ---        ┆ ---   │
│ i64        ┆ u32   │
╞════════════╪═══════╡
│ 1          ┆ 36461 │
│ 2          ┆ 48760 │
│ 3          ┆ 6154  │
│ 4          ┆ 473   │
│ 5          ┆ 1634  │
│ 6          ┆ 2989  │
│ 7          ┆ 3530  │
└────────────┴───────┘


## 4. Leakage-safe pipeline

The wilderness/soil columns are already one-hot (0/1) in the source data,
so only the 10 quantitative columns need scaling.

In [6]:
def summarize(name, metrics):
    m = metrics["modeling"]
    report = m.get("splits", {}).get("test") if isinstance(m.get("splits"), dict) else None
    if report is not None:
        acc = report.metrics.get("accuracy")
        f1 = report.metrics.get("f1_macro") or report.metrics.get("f1_weighted")
    else:
        acc, f1 = m.get("test", {}).get("accuracy"), m.get("test", {}).get("f1_macro")
    print(f"{name:20s} accuracy={acc:.4f}" + (f"  f1={f1:.4f}" if f1 is not None else ""))

In [7]:
base_preprocessing = [
    {
        "name": "split",
        "transformer": "TrainTestSplitter",
        "params": {"target_column": TARGET, "test_size": 0.2, "random_state": 42, "stratify": True},
    },
    {
        "name": "scale_numeric",
        "transformer": "StandardScaler",
        "params": {"columns": QUANTITATIVE},
    },
]

rf_config = {
    "preprocessing": base_preprocessing,
    "modeling": {
        "type": "random_forest_classifier",
        "params": {"n_estimators": 200, "max_depth": 20, "random_state": 42, "n_jobs": -1},
    },
}
rf_pipeline = SkyulfPipeline(rf_config)
rf_metrics = rf_pipeline.fit(model_df, target_column=TARGET)
summarize("Random Forest", rf_metrics)

Random Forest        accuracy=0.8576  f1=0.8539


## 5. Ensemble: `voting_classifier` (Random Forest + Extra Trees + HistGradientBoosting)

In [8]:
ensemble_config = {
    "preprocessing": base_preprocessing,
    "modeling": {
        "type": "voting_classifier",
        "params": {
            "base_estimators": ["random_forest", "extra_trees", "hist_gradient_boosting"],
            "voting": "soft",
        },
    },
}
ensemble_pipeline = SkyulfPipeline(ensemble_config)
ensemble_metrics = ensemble_pipeline.fit(model_df, target_column=TARGET)
summarize("Voting ensemble", ensemble_metrics)

Voting ensemble      accuracy=0.8894  f1=0.8877


## 6. Compare

In [9]:
summarize("Random Forest", rf_metrics)
summarize("Voting ensemble", ensemble_metrics)

Random Forest        accuracy=0.8576  f1=0.8539
Voting ensemble      accuracy=0.8894  f1=0.8877


## Key takeaways

- Public benchmarks for this dataset on the full 581k rows typically reach
  **~95-97% accuracy** with tuned tree ensembles; our 100k-row sample with
  untuned defaults gets meaningfully close, which is the expected trade-off
  for notebook runtime.
- Ensembling three different tree-based learners (bagged, extremely
  randomized, and boosted) usually edges out a single Random Forest,
  especially on the harder minority classes.
- What to try next: train on the full 581k rows (bundled, just slower),
  add a `stacking_classifier` with a linear meta-learner, and tune
  `max_depth`/`n_estimators` via the pipeline's `hyperparameter_tuner`.